In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import re
import json
import random
import time
from collections import Counter

# Project root
PROJECT_ROOT = Path.cwd().parent

# Allow importing mock_llm.py from project root
sys.path.insert(0, str(PROJECT_ROOT))

# Check required files
required_files = [
    PROJECT_ROOT / "data" / "raw" / "tickets.csv",
    PROJECT_ROOT / "mock_llm.py",
    PROJECT_ROOT / "outputs" / "customer_segments.csv"
]

for file in required_files:
    print(file.name, ":", "FOUND" if file.exists() else "NOT FOUND")

print("\nProject root:", PROJECT_ROOT)

tickets.csv : FOUND
mock_llm.py : FOUND
customer_segments.csv : FOUND

Project root: c:\Users\kamali\OneDrive\Documents\captone


Load & Audit Support Tickets

In [2]:
# Import supplied mock LLM
import mock_llm

# Load tickets
tickets = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "tickets.csv"
)

# Check dataset
print("Dataset shape:", tickets.shape)
print("\nColumns:")
print(tickets.columns.tolist())

# Check empty tickets
empty_count = tickets["text"].fillna("").str.strip().eq("").sum()

# Estimate tokens for every ticket
tickets["estimated_tokens"] = tickets["text"].fillna("").apply(
    mock_llm.est_tokens
)

# Check long tickets
long_count = (
    tickets["estimated_tokens"] > 400
).sum()

print("\nEmpty tickets:", empty_count)
print("Long tickets:", long_count)

print("\nEstimated tokens of long tickets:")
display(
    tickets.loc[
        tickets["estimated_tokens"] > 400,
        ["ticket_id", "estimated_tokens"]
    ]
)

Dataset shape: (164, 5)

Columns:
['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']

Empty tickets: 4
Long tickets: 4

Estimated tokens of long tickets:


,ticket_id,estimated_tokens
10,T0163,734
78,T0162,734
143,T0164,734
158,T0161,734


PII redaction

In [3]:
# Keep an untouched copy of the original ticket text
tickets["original_text"] = tickets["text"].fillna("")

# Create a separate column for redacted text
tickets["redacted_text"] = tickets["original_text"]

# 1. Redact email addresses
email_pattern = r"[\w.+-]+@[\w-]+\.[\w.]+"
tickets["redacted_text"] = tickets["redacted_text"].apply(
    lambda text: re.sub(email_pattern, "[EMAIL]", text)
)

# 2. Redact phone numbers
phone_pattern = r"\b[6-9]\d{9}\b"
tickets["redacted_text"] = tickets["redacted_text"].apply(
    lambda text: re.sub(phone_pattern, "[PHONE]", text)
)

# 3. Redact order IDs
order_pattern = r"ORD-\d{6}"
tickets["redacted_text"] = tickets["redacted_text"].apply(
    lambda text: re.sub(order_pattern, "[ORDER_ID]", text)
)

# Count redactions in the original text
email_count = tickets["original_text"].apply(
    lambda text: len(re.findall(email_pattern, text))
).sum()

phone_count = tickets["original_text"].apply(
    lambda text: len(re.findall(phone_pattern, text))
).sum()

order_count = tickets["original_text"].apply(
    lambda text: len(re.findall(order_pattern, text))
).sum()

print("Email redactions:", email_count)
print("Phone redactions:", phone_count)
print("Order ID redactions:", order_count)
print("Total redactions:", email_count + phone_count + order_count)

# Verify that no original PII patterns remain
remaining_emails = tickets["redacted_text"].apply(
    lambda text: len(re.findall(email_pattern, text))
).sum()

remaining_phones = tickets["redacted_text"].apply(
    lambda text: len(re.findall(phone_pattern, text))
).sum()

remaining_orders = tickets["redacted_text"].apply(
    lambda text: len(re.findall(order_pattern, text))
).sum()

print("\nRemaining emails:", remaining_emails)
print("Remaining phones:", remaining_phones)
print("Remaining order IDs:", remaining_orders)

assert (email_count, phone_count, order_count) == (55, 91, 132)
assert remaining_emails == 0
assert remaining_phones == 0
assert remaining_orders == 0

print("\nPII REDACTION CHECK PASSED")

Email redactions: 55
Phone redactions: 91
Order ID redactions: 132
Total redactions: 278

Remaining emails: 0
Remaining phones: 0
Remaining order IDs: 0

PII REDACTION CHECK PASSED


Input guardrails

In [4]:
# Apply input guardrails
tickets["status"] = "ready"

# Rule 1: Empty tickets
empty_mask = tickets["original_text"].str.strip().eq("")
tickets.loc[empty_mask, "status"] = "skipped_empty"

# Rule 2: Long tickets (> 400 estimated tokens)
long_mask = tickets["estimated_tokens"] > 400
tickets.loc[long_mask, "status"] = "flagged_long"

# Check status counts
print(tickets["status"].value_counts())

# Verify the guardrails
assert (tickets["status"] == "skipped_empty").sum() == 4
assert (tickets["status"] == "flagged_long").sum() == 4
assert (tickets["status"] == "ready").sum() == 156

# Confirm no empty or long ticket is marked ready
assert not tickets.loc[empty_mask, "status"].eq("ready").any()
assert not tickets.loc[long_mask, "status"].eq("ready").any()

print("\nINPUT GUARDRAILS PASSED")

status
ready            156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64

INPUT GUARDRAILS PASSED


Create zero-shot and few-shot prompts

In [5]:
# Allowed ticket categories
ALLOWED_CATEGORIES = [
    "billing",
    "delivery",
    "technical",
    "fraud"
]

# Zero-shot system prompt: no examples
ZERO_SHOT_PROMPT = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

Return ONLY valid JSON in this exact format:
{"category": "...", "reason": "..."}

Do not include markdown, code fences, or additional text.
"""

# Few-shot system prompt: includes examples
FEW_SHOT_PROMPT = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

EXAMPLES:

Ticket: I was charged twice for the same order.
Output: {"category": "billing", "reason": "Duplicate charge reported."}

Ticket: My order has not arrived even though the delivery date has passed.
Output: {"category": "delivery", "reason": "Order delivery is delayed."}

Ticket: Someone used my account without permission and placed an order.
Output: {"category": "fraud", "reason": "Possible unauthorized account activity."}

Return ONLY valid JSON in this exact format:
{"category": "...", "reason": "..."}

Do not include markdown, code fences, or additional text.
"""

# Validate prompt requirements
assert "CLASSIFY" in ZERO_SHOT_PROMPT
assert "EXAMPLES:" not in ZERO_SHOT_PROMPT

assert "CLASSIFY" in FEW_SHOT_PROMPT
assert "EXAMPLES:" in FEW_SHOT_PROMPT
assert "fraud" in FEW_SHOT_PROMPT.lower()

print("Zero-shot prompt created.")
print("Few-shot prompt created.")
print("\nPROMPT CHECKS PASSED")

Zero-shot prompt created.
Few-shot prompt created.

PROMPT CHECKS PASSED


Inspect mock_llm.py

In [6]:
# Display the supplied mock LLM code
import inspect

print(inspect.getsource(mock_llm))

"""mock_llm.py -- deterministic, offline stand-in for a hosted LLM chat API (no key, no network, no cost).
Interface mirrors a hosted chat API:  response = chat(messages, tools=None)
    response.choices[0].message.content / .tool_calls ; response.choices[0].finish_reason ; response.usage
Three behaviours are selected by a marker word in the FIRST (system) message:
    "CLASSIFY"  -> support-ticket classifier       "POLICY ASSISTANT" -> grounded policy answerer
    "AGENT"     -> refund agent that requests tools (ReAct style)"""
import json, math, re
from types import SimpleNamespace as NS

class TransientLLMError(Exception):
    """Simulated temporary failure (timeout / server busy). Retrying the same call succeeds."""

_state = {"calls": 0, "ok": 0}      # call counters that drive the simulated misbehaviour

def reset():
    """Restart the counters. Call once before each independent batch run."""
    _state["calls"] = 0; _state["ok"] = 0

CATS = ["billing", "delivery", "technical", "

Create the retry function

In [7]:
BASE_DELAY = 0.05
MAX_RETRIES = 3

retry_logs = []

def call_llm_with_retry(messages, ticket_id):
    for retry_number in range(MAX_RETRIES + 1):
        try:
            response = mock_llm.chat(messages)
            return response

        except mock_llm.TransientLLMError:
            if retry_number == MAX_RETRIES:
                print(f"Ticket {ticket_id}: failed after maximum retries.")
                raise

            wait_time = (
                BASE_DELAY
                * (2 ** retry_number)
                * random.uniform(0.9, 1.15)
            )

            retry_logs.append({
                "ticket_id": ticket_id,
                "retry_number": retry_number,
                "wait_seconds": wait_time
            })

            print(
                f"Ticket {ticket_id} | "
                f"Retry {retry_number + 1}/{MAX_RETRIES} | "
                f"Wait: {wait_time:.4f} seconds"
            )

            time.sleep(wait_time)

print("Retry function created.")
print("Maximum retries:", MAX_RETRIES)
print("Base delay:", BASE_DELAY)

Retry function created.
Maximum retries: 3
Base delay: 0.05


Validate LLM responses   

In [8]:
def validate_llm_output(content):
    if not content:
        return {
            "category": None,
            "reason": None,
            "status": "flagged_invalid_output"
        }

    # First: Try JSON validation
    try:
        data = json.loads(content)

        if not isinstance(data, dict):
            raise ValueError("JSON must be a dictionary")

        category = data.get("category")
        reason = data.get("reason")

        if category in ALLOWED_CATEGORIES and isinstance(reason, str):
            return {
                "category": category,
                "reason": reason,
                "status": "classified_json"
            }

        # Valid JSON but unsupported category:
        # Do not apply keyword fallback.
        return {
            "category": None,
            "reason": None,
            "status": "flagged_invalid_output"
        }

    except (json.JSONDecodeError, ValueError):
        pass

    # Second: Keyword fallback for non-JSON output
    words = re.findall(r"[a-z]+", content.lower())

    found_categories = [
        category for category in ALLOWED_CATEGORIES
        if category in words
    ]

    if len(found_categories) == 1:
        return {
            "category": found_categories[0],
            "reason": "Recovered using keyword fallback",
            "status": "classified_keyword"
        }

    # No category or multiple categories
    return {
        "category": None,
        "reason": None,
        "status": "flagged_invalid_output"
    }


print("JSON validation function created.")   

JSON validation function created.


Test the validation function

In [9]:
test_outputs = [
    '{"category": "billing", "reason": "Duplicate charge"}',
    'The customer has a fraud-related problem.',
    '{"category": "shipping", "reason": "Parcel movement"}',
    'The issue may involve billing and delivery.'
]

for output in test_outputs:
    print(output)
    print(validate_llm_output(output))
    print("-" * 60)

{"category": "billing", "reason": "Duplicate charge"}
{'category': 'billing', 'reason': 'Duplicate charge', 'status': 'classified_json'}
------------------------------------------------------------
The customer has a fraud-related problem.
{'category': 'fraud', 'reason': 'Recovered using keyword fallback', 'status': 'classified_keyword'}
------------------------------------------------------------
{"category": "shipping", "reason": "Parcel movement"}
{'category': None, 'reason': None, 'status': 'flagged_invalid_output'}
------------------------------------------------------------
The issue may involve billing and delivery.
{'category': None, 'reason': None, 'status': 'flagged_invalid_output'}
------------------------------------------------------------


Zero-shot classification run

In [10]:
# Reset mock LLM immediately before this independent run
mock_llm.reset()

# Reset retry logs for this run
retry_logs = []

zero_shot_results = []

for _, ticket in tickets.iterrows():

    result = {
        "ticket_id": ticket["ticket_id"],
        "customer_id": ticket["customer_id"],
        "true_category": ticket["true_category"],
        "status": ticket["status"],
        "category": None,
        "reason": None,
        "tokens_used": 0
    }

    # Only ready tickets can reach the model
    if ticket["status"] == "ready":

        messages = [
            {"role": "system", "content": ZERO_SHOT_PROMPT},
            {"role": "user", "content": ticket["redacted_text"]}
        ]

        response = call_llm_with_retry(
            messages,
            ticket["ticket_id"]
        )

        content = response.choices[0].message.content
        validation = validate_llm_output(content)

        result["category"] = validation["category"]
        result["reason"] = validation["reason"]
        result["status"] = validation["status"]
        result["tokens_used"] = response.usage.total_tokens

    zero_shot_results.append(result)

# Convert results into a DataFrame
zero_shot_df = pd.DataFrame(zero_shot_results)

print("\nZERO-SHOT STATUS COUNTS:")
print(zero_shot_df["status"].value_counts())

print("\nTotal tickets:", len(zero_shot_df))
print("Total model tokens:", zero_shot_df["tokens_used"].sum())
print("Total retry events:", len(retry_logs))

print("\nRetry log:")
display(pd.DataFrame(retry_logs))

# Check expected results
assert len(zero_shot_df) == 164
assert (zero_shot_df["status"] == "skipped_empty").sum() == 4
assert (zero_shot_df["status"] == "flagged_long").sum() == 4
assert zero_shot_df["tokens_used"].sum() > 0

print("\nZERO-SHOT RUN COMPLETED")

Ticket T0002 | Retry 1/3 | Wait: 0.0535 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0481 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0538 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0464 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0566 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0477 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0527 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0566 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0459 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0533 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0469 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0554 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0514 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0558 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0466 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0529 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0480 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0542 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0556 seconds

ZERO-SHOT STATUS COUNTS:
status
classified_json           135
classified_keyword       

,ticket_id,retry_number,wait_seconds
0,T0002,0,0.053510
1,T0116,0,0.048069
2,T0153,0,0.053834
3,T0039,0,0.046357
4,T0152,0,0.056639
5,T0043,0,0.047720
6,T0111,0,0.052745
7,T0155,0,0.056578
8,T0100,0,0.045862
9,T0068,0,0.053301



ZERO-SHOT RUN COMPLETED


Few-shot classification run

In [11]:
# Reset mock LLM immediately before this independent run
mock_llm.reset()
retry_logs = []

few_shot_results = []

for _, ticket in tickets.iterrows():

    result = {
        "ticket_id": ticket["ticket_id"],
        "customer_id": ticket["customer_id"],
        "true_category": ticket["true_category"],
        "status": ticket["status"],
        "category": None,
        "reason": None,
        "tokens_used": 0
    }

    if ticket["status"] == "ready":

        messages = [
            {"role": "system", "content": FEW_SHOT_PROMPT},
            {"role": "user", "content": ticket["redacted_text"]}
        ]

        response = call_llm_with_retry(
            messages,
            ticket["ticket_id"]
        )

        content = response.choices[0].message.content
        validation = validate_llm_output(content)

        result["category"] = validation["category"]
        result["reason"] = validation["reason"]
        result["status"] = validation["status"]
        result["tokens_used"] = response.usage.total_tokens

    few_shot_results.append(result)

few_shot_df = pd.DataFrame(few_shot_results)

print("\nFEW-SHOT STATUS COUNTS:")
print(few_shot_df["status"].value_counts())

print("\nTotal tickets:", len(few_shot_df))
print("Total model tokens:", few_shot_df["tokens_used"].sum())
print("Total retry events:", len(retry_logs))

# Verify results
assert len(few_shot_df) == 164
assert (few_shot_df["status"] == "skipped_empty").sum() == 4
assert (few_shot_df["status"] == "flagged_long").sum() == 4
assert few_shot_df["tokens_used"].sum() > 0

print("\nFEW-SHOT RUN COMPLETED")

Ticket T0002 | Retry 1/3 | Wait: 0.0555 seconds


Ticket T0116 | Retry 1/3 | Wait: 0.0495 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0573 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0517 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0497 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0505 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0491 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0540 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0454 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0454 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0512 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0527 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0474 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0478 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0509 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0540 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0548 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0499 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0573 seconds

FEW-SHOT STATUS COUNTS:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty 

Compare accuracy and confusion matrices

In [12]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Only usable classifications
valid_statuses = ["classified_json", "classified_keyword"]

zero_eval = zero_shot_df[
    zero_shot_df["status"].isin(valid_statuses)
].copy()

few_eval = few_shot_df[
    few_shot_df["status"].isin(valid_statuses)
].copy()

# Accuracy
zero_accuracy = accuracy_score(
    zero_eval["true_category"],
    zero_eval["category"]
)

few_accuracy = accuracy_score(
    few_eval["true_category"],
    few_eval["category"]
)

print("ZERO-SHOT ACCURACY:", round(zero_accuracy * 100, 2), "%")
print("FEW-SHOT ACCURACY:", round(few_accuracy * 100, 2), "%")

# Confusion matrices
labels = ALLOWED_CATEGORIES

zero_cm = confusion_matrix(
    zero_eval["true_category"],
    zero_eval["category"],
    labels=labels
)

few_cm = confusion_matrix(
    few_eval["true_category"],
    few_eval["category"],
    labels=labels
)

print("\nZERO-SHOT CONFUSION MATRIX")
display(pd.DataFrame(
    zero_cm,
    index=labels,
    columns=labels
))

print("\nFEW-SHOT CONFUSION MATRIX")
display(pd.DataFrame(
    few_cm,
    index=labels,
    columns=labels
))

# Classification reports
print("\nZERO-SHOT CLASSIFICATION REPORT")
print(classification_report(
    zero_eval["true_category"],
    zero_eval["category"],
    labels=labels,
    zero_division=0
))

print("\nFEW-SHOT CLASSIFICATION REPORT")
print(classification_report(
    few_eval["true_category"],
    few_eval["category"],
    labels=labels,
    zero_division=0
))

# Token comparison
print("\nTOKEN COMPARISON")
print("Zero-shot total:", zero_shot_df["tokens_used"].sum())
print("Few-shot total:", few_shot_df["tokens_used"].sum())

print(
    "Zero-shot tokens per classified ticket:",
    round(zero_shot_df["tokens_used"].sum() / len(zero_eval), 2)
)

print(
    "Few-shot tokens per classified ticket:",
    round(few_shot_df["tokens_used"].sum() / len(few_eval), 2)
)


ZERO-SHOT ACCURACY: 85.71 %
FEW-SHOT ACCURACY: 100.0 %

ZERO-SHOT CONFUSION MATRIX


,billing,delivery,technical,fraud
billing,38,0,0,0
delivery,0,36,0,0
technical,10,0,29,0
fraud,11,0,0,23



FEW-SHOT CONFUSION MATRIX


,billing,delivery,technical,fraud
billing,38,0,0,0
delivery,0,36,0,0
technical,0,0,39,0
fraud,0,0,0,34



ZERO-SHOT CLASSIFICATION REPORT
              precision    recall  f1-score   support

     billing       0.64      1.00      0.78        38
    delivery       1.00      1.00      1.00        36
   technical       1.00      0.74      0.85        39
       fraud       1.00      0.68      0.81        34

    accuracy                           0.86       147
   macro avg       0.91      0.86      0.86       147
weighted avg       0.91      0.86      0.86       147


FEW-SHOT CLASSIFICATION REPORT
              precision    recall  f1-score   support

     billing       1.00      1.00      1.00        38
    delivery       1.00      1.00      1.00        36
   technical       1.00      1.00      1.00        39
       fraud       1.00      1.00      1.00        34

    accuracy                           1.00       147
   macro avg       1.00      1.00      1.00       147
weighted avg       1.00      1.00      1.00       147


TOKEN COMPARISON
Zero-shot total: 12520
Few-shot total: 24220
Ze

Token budget demonstration

In [13]:
# Token budget demonstration: zero-shot
TOKEN_BUDGET = 6000

mock_llm.reset()
retry_logs = []

budget_results = []
running_tokens = 0

for _, ticket in tickets.iterrows():

    result = {
        "ticket_id": ticket["ticket_id"],
        "customer_id": ticket["customer_id"],
        "true_category": ticket["true_category"],
        "status": ticket["status"],
        "category": None,
        "reason": None,
        "tokens_used": 0
    }

    if ticket["status"] == "ready":

        # Check budget BEFORE making a model call
        if running_tokens >= TOKEN_BUDGET:
            result["status"] = "budget_exhausted"

        else:
            messages = [
                {"role": "system", "content": ZERO_SHOT_PROMPT},
                {"role": "user", "content": ticket["redacted_text"]}
            ]

            response = call_llm_with_retry(
                messages,
                ticket["ticket_id"]
            )

            content = response.choices[0].message.content
            validation = validate_llm_output(content)

            result["category"] = validation["category"]
            result["reason"] = validation["reason"]
            result["status"] = validation["status"]
            result["tokens_used"] = response.usage.total_tokens

            running_tokens += response.usage.total_tokens

    budget_results.append(result)

budget_demo_df = pd.DataFrame(budget_results)

# Save the demonstration output
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

budget_demo_path = output_dir / "triage_budget_demo.csv"
budget_demo_df.to_csv(budget_demo_path, index=False)

print("BUDGET STATUS COUNTS:")
print(budget_demo_df["status"].value_counts())

print("\nToken budget:", TOKEN_BUDGET)
print("Actual tokens used:", running_tokens)
print("Budget exhausted tickets:",
      (budget_demo_df["status"] == "budget_exhausted").sum())
print("Retry events:", len(retry_logs))
 
print("\nSaved to:", budget_demo_path)

assert len(budget_demo_df) == 164
assert running_tokens >= TOKEN_BUDGET
assert (budget_demo_df["status"] == "budget_exhausted").sum() > 0
assert (budget_demo_df["status"] == "skipped_empty").sum() == 4
assert (budget_demo_df["status"] == "flagged_long").sum() == 4

print("\nTOKEN BUDGET CHECK PASSED")

Ticket T0002 | Retry 1/3 | Wait: 0.0492 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0558 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0503 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0466 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0568 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0520 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0550 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0506 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0488 seconds
BUDGET STATUS COUNTS:
status
budget_exhausted          81
classified_json           66
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     4
Name: count, dtype: int64

Token budget: 6000
Actual tokens used: 6032
Budget exhausted tickets: 81
Retry events: 9

Saved to: c:\Users\kamali\OneDrive\Documents\captone\outputs\triage_budget_demo.csv

TOKEN BUDGET CHECK PASSED


Prepare customer segment information

In [14]:
customer_segments = pd.read_csv(
    PROJECT_ROOT / "outputs" / "customer_segments.csv"
)

print("Customer segments shape:", customer_segments.shape)
print("\nAvailable columns:")
print(customer_segments.columns.tolist())

display(customer_segments.head())

print("\nPremium-related columns:")
print([
    col for col in customer_segments.columns
    if "premium" in col.lower() or "segment" in col.lower()
])

Customer segments shape: (1175, 7)

Available columns:
['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,Occasional Customers,False
1,C0002,28,7,4644.90,1,Premium Customers,True
2,C0003,7,22,3644.85,2,Loyal Customers,False
3,C0004,0,5,369.00,0,Occasional Customers,False
4,C0005,147,2,324.00,3,At-Risk Customers,False



Premium-related columns:
['segment_id', 'segment_name', 'is_premium']


Final few-shot classification and ticket routing

In [15]:
# Final few-shot run with token budget
TOKEN_BUDGET = 100000

# Reset mock LLM immediately before the final run
mock_llm.reset()
retry_logs = []

final_results = []
running_tokens = 0

# Create customer segment lookup
segment_lookup = customer_segments.set_index("customer_id")[
    ["segment_name", "is_premium"]
].to_dict("index")

for _, ticket in tickets.iterrows():

    customer_id = ticket["customer_id"]

    # Get customer segment; missing customer defaults to non-premium
    customer_info = segment_lookup.get(customer_id, {})
    segment_name = customer_info.get("segment_name")
    is_premium = customer_info.get("is_premium", False)

    result = {
        "ticket_id": ticket["ticket_id"],
        "customer_id": customer_id,
        "status": ticket["status"],
        "category": None,
        "segment_name": segment_name,
        "priority": "NORMAL",
        "route": None,
        "tokens_used": 0
    }

    if ticket["status"] == "ready":

        # Check token budget before model call
        if running_tokens >= TOKEN_BUDGET:
            result["status"] = "budget_exhausted"

        else:
            messages = [
                {"role": "system", "content": FEW_SHOT_PROMPT},
                {"role": "user", "content": ticket["redacted_text"]}
            ]

            response = call_llm_with_retry(
                messages,
                ticket["ticket_id"]
            )

            content = response.choices[0].message.content
            validation = validate_llm_output(content)

            result["category"] = validation["category"]
            result["status"] = validation["status"]
            result["tokens_used"] = response.usage.total_tokens

            running_tokens += response.usage.total_tokens

    # Apply routing rules
    if result["status"] in [
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ]:
        result["route"] = "human_review"
        result["priority"] = "NORMAL"

    elif result["category"] == "fraud":
        result["route"] = "security_team"
        result["priority"] = "HIGH"

    elif is_premium:
        result["route"] = "premium_desk"
        result["priority"] = "HIGH"

    else:
        result["route"] = "support_queue"
        result["priority"] = "NORMAL"

    final_results.append(result)


# Create final DataFrame
triage_results = pd.DataFrame(final_results)

# Keep only required columns in exact order
required_columns = [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

triage_results = triage_results[required_columns]

# Save final CSV
final_output_path = PROJECT_ROOT / "outputs" / "triage_results.csv"

triage_results.to_csv(final_output_path, index=False)

# Display results
print("FINAL STATUS COUNTS:")
print(triage_results["status"].value_counts())

print("\nROUTING COUNTS:")
print(triage_results["route"].value_counts())

print("\nPRIORITY COUNTS:")
print(triage_results["priority"].value_counts())

print("\nTotal tickets:", len(triage_results))
print("Total tokens used:", running_tokens)
print("Retry events:", len(retry_logs))

print("\nSaved:", final_output_path)

# Validation checks
assert len(triage_results) == 164
assert list(triage_results.columns) == required_columns
assert triage_results["ticket_id"].is_unique
assert running_tokens <= TOKEN_BUDGET

assert (
    triage_results.loc[
        triage_results["category"] == "fraud", "route"
    ] == "security_team"
).all()

assert (
    triage_results.loc[
        triage_results["status"].isin([
            "skipped_empty",
            "flagged_long",
            "flagged_invalid_output",
            "budget_exhausted"
        ]),
        "route"
    ] == "human_review"
).all()

print("\nFINAL TRIAGE CHECKS PASSED")

Ticket T0002 | Retry 1/3 | Wait: 0.0513 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0537 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0502 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0536 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0529 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0490 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0505 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0553 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0499 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0560 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0573 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0511 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0456 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0477 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0452 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0502 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0492 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0450 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0538 seconds
FINAL STATUS COUNTS:
status
classified_json           135
classified_keyword         12


In [16]:
# Routing validation

expected_routes = {
    "support_queue": 102,
    "security_team": 34,
    "human_review": 17,
    "premium_desk": 11
}

actual_routes = triage_results["route"].value_counts().to_dict()

print("EXPECTED ROUTES:", expected_routes)
print("ACTUAL ROUTES:", actual_routes)

assert actual_routes == expected_routes
assert sum(actual_routes.values()) == 164

# Priority validation
expected_priorities = {
    "HIGH": 45,
    "NORMAL": 119
}

actual_priorities = (
    triage_results["priority"].value_counts().to_dict()
)

print("\nEXPECTED PRIORITIES:", expected_priorities)
print("ACTUAL PRIORITIES:", actual_priorities)

assert actual_priorities == expected_priorities

# Every ticket must have a valid route and priority
assert triage_results["route"].notna().all()
assert triage_results["priority"].isin(
    ["HIGH", "NORMAL"]
).all()

print("\nROUTING VALIDATION PASSED")

EXPECTED ROUTES: {'support_queue': 102, 'security_team': 34, 'human_review': 17, 'premium_desk': 11}
ACTUAL ROUTES: {'support_queue': 102, 'security_team': 34, 'human_review': 17, 'premium_desk': 11}

EXPECTED PRIORITIES: {'HIGH': 45, 'NORMAL': 119}
ACTUAL PRIORITIES: {'NORMAL': 119, 'HIGH': 45}

ROUTING VALIDATION PASSED


Verify the saved CSV

In [17]:
# Reload the saved CSV
verified_triage = pd.read_csv(final_output_path)

print("Saved CSV shape:", verified_triage.shape)

print("\nColumns:")
print(verified_triage.columns.tolist())

print("\nMissing values:")
print(verified_triage.isnull().sum())

print("\nFirst 10 records:")
display(verified_triage.head(10))

# Final file checks
assert verified_triage.shape == (164, 8)
assert verified_triage["ticket_id"].is_unique

assert verified_triage["route"].notna().all()
assert verified_triage["priority"].notna().all()

print("\nSAVED CSV VERIFICATION PASSED")

Saved CSV shape: (164, 8)

Columns:
['ticket_id', 'customer_id', 'status', 'category', 'segment_name', 'priority', 'route', 'tokens_used']

Missing values:
ticket_id        0
customer_id      0
status           0
category        17
segment_name     0
priority         0
route            0
tokens_used      0
dtype: int64

First 10 records:


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At-Risk Customers,HIGH,security_team,159
1,T0073,C0080,classified_json,billing,Loyal Customers,NORMAL,support_queue,155
2,T0095,C1087,classified_json,technical,Occasional Customers,NORMAL,support_queue,157
3,T0159,C1172,skipped_empty,NaN,Loyal Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional Customers,HIGH,security_team,159
5,T0124,C0139,classified_json,fraud,Loyal Customers,HIGH,security_team,159
6,T0061,C0812,classified_json,billing,Loyal Customers,NORMAL,support_queue,151
7,T0118,C0355,classified_json,delivery,Loyal Customers,NORMAL,support_queue,154
8,T0037,C1156,classified_json,billing,Premium Customers,HIGH,premium_desk,155
9,T0002,C0881,classified_json,delivery,Occasional Customers,NORMAL,support_queue,153



SAVED CSV VERIFICATION PASSED


In [18]:
# Final output validation

status_counts = verified_triage["status"].value_counts()

print("STATUS COUNTS:")
print(status_counts)

assert status_counts["classified_json"] == 135
assert status_counts["classified_keyword"] == 12
assert status_counts["flagged_invalid_output"] == 9

eligible_count = (
    status_counts["classified_json"]
    + status_counts["classified_keyword"]
    + status_counts["flagged_invalid_output"]
)

print("\nEligible tickets:", eligible_count)

assert eligible_count == 156
assert verified_triage["category"].isna().sum() == 17

print("\nFINAL OUTPUT VALIDATION PASSED")

STATUS COUNTS:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64

Eligible tickets: 156

FINAL OUTPUT VALIDATION PASSED


# Part C – Support Ticket Triage with Guardrails

## 1. Input Guardrails

A total of 164 support tickets were processed in their original file order. Input validation rules were applied before sending tickets for classification.

- **Empty Ticket Detection:** Tickets containing only whitespace or empty text were identified using whitespace stripping and marked as `skipped_empty`.
- **Long Ticket Detection:** Tickets with more than 400 estimated tokens were marked as `flagged_long`.
- Four empty tickets were skipped.
- Four long tickets (734 estimated tokens each) were flagged.
- The remaining 156 tickets were marked as `ready` for classification.

**Input Validation Summary:**

- Ready tickets: 156
- Skipped empty tickets: 4
- Flagged long tickets: 4
- Total tickets: 164

All input guardrail assertions passed. Empty and long tickets were excluded from model classification.

## 2. PII Redaction and Data Privacy

Personally Identifiable Information (PII) was redacted before sending ticket content to the model.

The original ticket text was preserved separately in the `original_text` column. A separate `redacted_text` column was created for processing, ensuring that the original data remained unchanged.

Regular expressions were used to identify and replace:

- Email Addresses: 55 replaced with `[EMAIL]`.
- Phone Numbers: 91 replaced with `[PHONE]`.
- Order IDs: 132 replaced with `[ORDER_ID]`.
- Total PII instances redacted: 278.

Validation confirmed that no matching email, phone number, or order ID patterns remained in the redacted text.

The model received only the redacted ticket text, while the original text remained available separately for reference.

**PII Redaction Validation: PASSED**

## 3. Prompting Strategy

Zero-shot and few-shot prompting strategies were compared using four predefined support ticket categories:

- Billing
- Delivery
- Technical
- Fraud

Both prompts instructed the model to return JSON-only responses containing the classification result.

- **Zero-shot Prompting:** The model classified tickets without being provided with examples.
- **Few-shot Prompting:** The model received sample classification examples to guide its predictions.

The objective was to evaluate the impact of examples on classification accuracy and token consumption.

## 4. Retry Handling and Error Recovery

A retry mechanism using exponential backoff with random jitter was implemented to handle transient model errors.

**Retry Configuration:**

- Initial Delay: 0.05 seconds.
- Maximum Retries: 3.
- Total Attempts: 4 (1 initial attempt + 3 retries).
- Backoff Strategy: Exponential Backoff.
- Jitter Multiplier: Random value between 0.9 and 1.15.
- Maximum Wait per Retry: 0.23 seconds.

The waiting time doubled after each failed attempt, with a random jitter factor applied to reduce repeated request timing.

Retry events were logged with the ticket ID, retry number, and actual waiting time.

All 19 simulated transient errors were successfully recovered during the full classification runs.

## 5. Output Validation

Model responses were validated to ensure that only supported classification categories were accepted.

The validation process included:

- Valid JSON responses were accepted and recorded as `classified_json`.
- Non-JSON responses containing exactly one supported category keyword were handled using keyword fallback and recorded as `classified_keyword`.
- Invalid JSON responses, unsupported categories, and ambiguous outputs were marked as `flagged_invalid_output` and routed for human review.

**Output Validation Summary:**

- Valid JSON Classification: 135 tickets.
- Keyword Fallback Classification: 12 tickets.
- Invalid Model Output: 9 tickets.
- Total Eligible Tickets: 156.

The calculation is:

135 + 12 + 9 = 156 eligible tickets.

The remaining eight tickets consisted of four empty and four long tickets, which were excluded before classification.

The 9 invalid outputs were not treated as successful classifications and were routed to human review.

## 6. Zero-shot vs Few-shot Comparison

Both prompting strategies were evaluated using 147 usable classifications. The nine invalid outputs were excluded from the accuracy calculation.

**Evaluation Results:**

- Usable Classifications: 147 for both strategies.
- Zero-shot Accuracy: 85.71%.
- Few-shot Accuracy: 100%.
- Zero-shot Total Tokens Used: 12,520.
- Few-shot Total Tokens Used: 24,220.
- Zero-shot Tokens per Usable Classification: 85.17.
- Few-shot Tokens per Usable Classification: 164.76.

**Key Observations:**

- Zero-shot prompting achieved 85.71% accuracy.
- Few-shot prompting achieved 100% accuracy on the 147 usable classifications.
- Few-shot prompting improved accuracy by 14.29 percentage points.
- However, few-shot prompting consumed approximately 93.5% more tokens than zero-shot prompting.
- Zero-shot classification incorrectly predicted 11 fraud tickets and 10 technical tickets as billing.

Few-shot prompting demonstrated higher classification accuracy in this evaluation, while zero-shot prompting required fewer tokens.

**Evaluation Limitation:** The reported 100% few-shot accuracy applies only to the 147 usable classifications, not to all 164 input tickets. It should not be interpreted as a guarantee of perfect performance on unseen tickets.

## 7. Token Budget Management

Token budget controls were implemented to demonstrate how model usage can be restricted.

### 7.1 Zero-shot Budget Demonstration

A token budget of 6,000 was configured for the zero-shot classification demonstration.

**Budget Results:**

- Configured Token Budget: 6,000.
- Actual Tokens Used: 6,032.
- Budget Overshoot: 32 tokens.
- Budget Exhausted Tickets: 81.

The actual usage exceeded the configured budget by 32 tokens because the budget was checked before each model call. Tokens from the completed response were added only after that call finished.

As a result, a single model response could cause the accumulated usage to exceed the remaining budget.

A total of 81 tickets were marked as `budget_exhausted` and were not sent for further classification.

**Limitation:** The pre-call budget check does not guarantee a strict token cap. A stricter implementation would need to account for the maximum expected response usage before making each model call.

### 7.2 Final Few-shot Run

The final few-shot classification and routing process used a token budget of 100,000.

- Configured Budget: 100,000 tokens.
- Actual Token Consumption: 24,220 tokens.
- The final run completed within the configured budget.

## 8. Final Routing Rules and Validation

After classification, tickets were assigned to appropriate support teams based on their classification status, category, and customer segment.

**Routing Rules:**

- Fraud Ticket: Assigned to `security_team` with HIGH priority.
- Valid Ticket from a Premium Customer: Assigned to `premium_desk` with HIGH priority.
- Other Valid Ticket: Assigned to `support_queue` with NORMAL priority.
- Empty Ticket: Assigned to `human_review` with NORMAL priority.
- Long Ticket: Assigned to `human_review` with NORMAL priority.
- Invalid Model Output: Assigned to `human_review` with NORMAL priority.
- Budget-exhausted Ticket: Assigned to `human_review` with NORMAL priority.

**Final Routing Distribution:**

- `support_queue`: 102 tickets.
- `security_team`: 34 tickets.
- `human_review`: 17 tickets.
- `premium_desk`: 11 tickets.
- Total: 164 tickets.

**Priority Distribution:**

- HIGH: 45 tickets.
- NORMAL: 119 tickets.
- Total: 164 tickets.

Routing validation confirmed that all 164 tickets received a valid route and priority.

The actual route counts and priority counts matched the expected values. All routing assertions passed.

**Routing Validation: PASSED**

## 9. Final Output and File Verification

The final results were saved in CSV format under the project's `outputs` directory.

**Generated Files:**

- `outputs/triage_budget_demo.csv` – Contains results from the zero-shot token budget demonstration.
- `outputs/triage_results.csv` – Contains the final few-shot classification and ticket routing results.

**Final CSV Structure:**

- Total Records: 164.
- Total Columns: 8.
- Unique Ticket IDs: Yes.
- Missing Route Values: 0.
- Missing Priority Values: 0.
- Missing Segment Names: 0.
- Missing Category Values: 17.

The final output contains the following eight columns in the required order:

1. `ticket_id`
2. `customer_id`
3. `status`
4. `category`
5. `segment_name`
6. `priority`
7. `route`
8. `tokens_used`

The 17 missing category values are expected because they correspond to nine invalid model outputs, four skipped empty tickets, and four flagged long tickets. These tickets were routed to human review rather than being assigned an unsupported classification.

The saved CSV was reloaded and verified successfully. Record count, column structure, ticket ID uniqueness, route values, and priority values passed the validation checks.

**Final Output Validation: PASSED**

## 10. Conclusion

Part C implemented an end-to-end support ticket triage workflow with input guardrails, PII redaction, zero-shot and few-shot prompting, retry handling, output validation, token budget management, and customer-segment-based ticket routing.

The workflow processed 164 tickets, preserved original ticket text, protected sensitive information before model calls, and routed tickets according to predefined business rules.

The comparison showed that few-shot prompting achieved higher accuracy on the usable evaluation set, while requiring greater token consumption. The final routing process successfully assigned all tickets to the appropriate support queues or human review.

All major input, privacy, retry, routing, and output validation checks passed.